In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.eval_crew import load_cases

cases = load_cases(str(PROJECT_ROOT) + "/data/sample/lab13_test_cases.jsonl")
len(cases), cases[0]


(10,
 {'case_id': 'case_001',
  'input': 'Кримінальне провадження №42016000000002788 від 08.10.2016 передано до суду, сума збитків 5000 грн.',
  'expected_behavior': 'extract document id, CASE_ID, date, amount and UAH',
  'gold': {'document_id': '42016000000002788',
   'document_type': 'CASE_ID',
   'date_iso': '2016-10-08',
   'date_text': '08.10.2016',
   'amount_value': 5000.0,
   'amount_currency': 'UAH'},
  'error_category': 'simple_valid'})

In [3]:
from src.agents import GroqLLMClient, TriagerAgent, ExtractorAgent, ReviewerAgent, RepairAgent
from src.llm_extract import DEFAULT_MODEL

client = GroqLLMClient(model=DEFAULT_MODEL)
agents = {
    "triager": TriagerAgent(client),
    "extractor": ExtractorAgent(client),
    "reviewer": ReviewerAgent(client),
    "repair": RepairAgent(client),
}
list(agents)


['triager', 'extractor', 'reviewer', 'repair']

In [4]:
DELEGATION_RULES = [
    "Triager is always called first.",
    "Extractor receives source input plus triage route/schema.",
    "Reviewer always checks Extractor output.",
    "verdict=accept -> final output.",
    "verdict!=accept -> RepairAgent, then rule-based fallback if repair fails.",
    "unsafe or unresolved output -> manual review status.",
]
DELEGATION_RULES


['Triager is always called first.',
 'Extractor receives source input plus triage route/schema.',
 'Reviewer always checks Extractor output.',
 'verdict=accept -> final output.',
 'verdict!=accept -> RepairAgent, then rule-based fallback if repair fails.',
 'unsafe or unresolved output -> manual review status.']

In [5]:
from src.eval_crew import run_single_agent_baseline

baseline_runs = run_single_agent_baseline(cases, model=DEFAULT_MODEL)
baseline_runs[0]


BaselineRun(case_id='case_001', raw_output='{\n  "document_id": "42016000000002788",\n  "document_type": "CASE_ID",\n  "date_iso": null,\n  "date_text": "08.10.2016",\n  "amount_value": 5000,\n  "amount_currency": "UAH"\n}', valid=False, errors=['Consistency error: date_text must be null when date_iso is null'], prediction=None)

In [6]:
from src.eval_crew import run_crew

crew_runs = run_crew(cases, model=DEFAULT_MODEL)
crew_runs[0]


CrewRun(case_id='case_001', input='Кримінальне провадження №42016000000002788 від 08.10.2016 передано до суду, сума збитків 5000 грн.', triager_output={'task_type': 'document', 'route': 'document_signal_schema', 'expected_fields': {'document_id': None, 'document_type': 'CASE_ID', 'date_iso': None, 'date_text': None, 'amount_value': 5000, 'amount_currency': 'UAH'}, 'difficulty': 2, 'special_handling': False, 'notes': ''}, extractor_output={'document_id': None, 'document_type': 'CASE_ID', 'date_iso': None, 'date_text': '08.10.2016', 'amount_value': 5000, 'amount_currency': 'UAH'}, reviewer_output={'verdict': 'fallback_needed', 'valid_json': True, 'schema_ok': True, 'consistency_ok': False, 'issues': [{'field': 'document_type', 'problem': 'document_type present without document_id', 'severity': 'critical'}, {'field': 'date_text', 'problem': 'date_text present without normalized date_iso', 'severity': 'medium'}, {'field': 'document_id', 'problem': 'likely missed field according to rule evi

In [7]:
first = crew_runs[0]
first.reviewer_output


{'verdict': 'fallback_needed',
 'valid_json': True,
 'schema_ok': True,
 'consistency_ok': False,
 'issues': [{'field': 'document_type',
   'problem': 'document_type present without document_id',
   'severity': 'critical'},
  {'field': 'date_text',
   'problem': 'date_text present without normalized date_iso',
   'severity': 'medium'},
  {'field': 'document_id',
   'problem': 'likely missed field according to rule evidence',
   'severity': 'medium'},
  {'field': 'date_iso',
   'problem': 'likely missed field according to rule evidence',
   'severity': 'medium'},
  {'field': '<llm_review>',
   'problem': 'Consistency error: document_type must be null when document_id is null',
   'severity': 'medium'},
  {'field': '<llm_review>',
   'problem': 'Consistency error: date_text must be null when date_iso is null',
   'severity': 'medium'}],
 'recommended_action': 'run_repair_then_rule_fallback',
 'validator_errors': ['Consistency error: document_type must be null when document_id is null',
 

In [8]:
fallback_cases = [run for run in crew_runs if run.fallback_triggered]
[(run.case_id, run.status, run.fallback_output.get("action") if run.fallback_output else None) for run in fallback_cases]


[('case_001', 'accepted_after_repair', 'repair_agent'),
 ('case_002', 'accepted_after_repair', 'repair_agent'),
 ('case_003', 'partial_manual_review', 'rule_based_partial'),
 ('case_004', 'partial_manual_review', 'rule_based_partial'),
 ('case_005', 'failed_manual_review', 'safe_failure'),
 ('case_006', 'accepted_after_repair', 'repair_agent'),
 ('case_007', 'accepted_after_repair', 'repair_agent'),
 ('case_008', 'partial_manual_review', 'rule_based_partial'),
 ('case_009', 'partial_manual_review', 'rule_based_partial'),
 ('case_010', 'failed_manual_review', 'safe_failure')]

In [9]:
[(run.case_id, run.status, run.final_output.get("needs_manual_review")) for run in crew_runs]


[('case_001', 'accepted_after_repair', False),
 ('case_002', 'accepted_after_repair', False),
 ('case_003', 'partial_manual_review', True),
 ('case_004', 'partial_manual_review', True),
 ('case_005', 'failed_manual_review', True),
 ('case_006', 'accepted_after_repair', False),
 ('case_007', 'accepted_after_repair', False),
 ('case_008', 'partial_manual_review', True),
 ('case_009', 'partial_manual_review', True),
 ('case_010', 'failed_manual_review', True)]

In [10]:
from src.crew_workflow import save_crew_logs

logs_path = str(PROJECT_ROOT / "docs/crew_logs_lab13.jsonl")
save_crew_logs(logs_path, crew_runs)
logs_path


'docs/crew_logs_lab13.jsonl'

In [11]:
from src.eval_crew import compute_metrics

metrics = compute_metrics(cases, baseline_runs, crew_runs)
metrics


{'total_cases': 10,
 'single_agent_valid_output_rate': 0.5,
 'crew_valid_final_output_rate': 0.9,
 'reviewer_catch_rate': 1.0,
 'fallback_activation_rate': 1.0,
 'fallback_success_rate': 0.8,
 'manual_review_rate': 0.6,
 'hallucination_issue_count': 0,
 'missing_required_field_issue_count': 11,
 'average_agents_called_per_case': 4,
 'average_repair_attempts_per_case': 1}

In [13]:
import pandas as pd
from src.eval_crew import build_error_analysis

error_analysis = build_error_analysis(cases, crew_runs)
pd.DataFrame(error_analysis)[["case_id", "error_category", "fallback_action", "possible_fix"]]


,case_id,error_category,fallback_action,possible_fix
0,case_001,simple_valid,repair_agent,Inspect reviewer issues and add a targeted rul...
1,case_002,missing_required_signal,repair_agent,Inspect reviewer issues and add a targeted rul...
2,case_003,ambiguous_entity,rule_based_partial,Tighten grounding rules for document_id and re...
3,case_004,relative_date,rule_based_partial,Add optional current-date-aware normalization ...
4,case_005,hallucination_prone,safe_failure,Inspect reviewer issues and add a targeted rul...
5,case_006,noisy_typos,repair_agent,Inspect reviewer issues and add a targeted rul...
6,case_007,fallback_required,repair_agent,"Keep rule-based fallback for short IDs, dates,..."
7,case_008,reviewer_rejection,rule_based_partial,Tighten grounding rules for document_id and re...
8,case_009,repair_succeeds,rule_based_partial,"Keep rule-based fallback for short IDs, dates,..."
9,case_010,manual_review_after_failed_repair,safe_failure,Return safe partial output and route to manual...


In [14]:
from src.eval_crew import write_audit_summary

audit_path = PROJECT_ROOT / "docs/audit_summary_lab13.md"
write_audit_summary(audit_path, metrics, cases, crew_runs, error_analysis)
audit_path.read_text(encoding="utf-8")[:1000]


'# Lab 13 Audit Summary\n\n## 1. Use case\n- Multi-agent document signal extraction for Ukrainian legal/news text.\n- Fields: `document_id`, `document_type`, `date_iso`, `date_text`, `amount_value`, `amount_currency`.\n\n## 2. Agents implemented\n- Triager: selects route, expected fields, difficulty and special handling.\n- Extractor: returns schema-only JSON.\n- Reviewer: checks JSON validity, schema, grounding and consistency.\n- Repair/Fallback: repairs reviewer issues and applies rule-based partial extraction when repair fails.\n\n## 3. Test cases: `10`\n\n## 4. Metrics\n- Valid final output rate: `90.00%`\n- Reviewer catch rate: `100.00%`\n- Fallback activation rate: `100.00%`\n- Fallback success rate: `80.00%`\n- Manual review rate: `60.00%`\n\n## 5. Single-agent vs crew comparison\n| Variant | Valid output rate | Notes |\n|---|---:|---|\n| Single-agent baseline | 50.00% | One Groq extraction call, no independent review. |\n| Multi-agent crew | 90.00% | Triager + Extractor + Revi